In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

!pip install gdown
!gdown --id 1ApDpf57o0smumnPEd__tpIpfxplYpbxz
!mkdir -p data/uncleaned-data/
!unzip -o uncleaned-data.zip -d data/uncleaned-data/

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ApDpf57o0smumnPEd__tpIpfxplYpbxz
From (redirected): https://drive.google.com/uc?id=1ApDpf57o0smumnPEd__tpIpfxplYpbxz&confirm=t&uuid=8e9aa835-f80d-41f7-9d53-fed53645dc97
To: /content/uncleaned-data.zip
100% 28.2M/28.2M [00:00<00:00, 172MB/s]
Archive:  uncleaned-data.zip
   creating: data/uncleaned-data/uncleaned-data/
  inflating: data/uncleaned-data/uncleaned-data/customers.csv  
  inflating: data/uncleaned-data/uncleaned-data/geography.csv  
  inflating: data/uncleaned-data/uncleaned-data/inventory.csv  
  inflating: data/uncleaned-data/uncleaned-data/orders.csv  
  inflating: data/uncleaned-data/uncleaned-data/order_items.csv  
  inflating: data/uncleaned-data/uncleaned-data/payments.csv  
  inflating: d

In [ ]:
"""
DATA PREPROCESSING
Handles data ingestion, type casting, missing value imputation, and standardization.
Designed for reproducibility in ML forecasting pipelines.
"""

import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ----------------------------------------------------------------------------
# 1. DIRECTORY SETUP & DATA LOADING
# ----------------------------------------------------------------------------
INPUT_DIR = 'data/uncleaned-data/'
OUTPUT_DIR = 'data/cleaned-data/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

sheet_names = [
    'returns', 'promotions', 'products', 'payments', 'customers',
    'geography', 'inventory', 'order_items', 'web_traffic',
    'shipments', 'sales', 'reviews', 'orders'
]

dfs = {}
print("[INFO] Starting data ingestion process...")

for sheet in sheet_names:
    file_path = f"{INPUT_DIR}{sheet}.csv"
    if os.path.exists(file_path):
        dfs[sheet] = pd.read_csv(file_path)
        print(f"Loaded: {sheet.ljust(15)} | Shape: {dfs[sheet].shape}")
    else:
        print(f"[WARNING] File not found: {file_path}")

[INFO] Starting data ingestion process...
Loaded: returns         | Shape: (39939, 7)
Loaded: promotions      | Shape: (50, 10)
Loaded: products        | Shape: (2412, 8)
Loaded: payments        | Shape: (646945, 4)
Loaded: customers       | Shape: (121930, 7)
Loaded: geography       | Shape: (39948, 4)
Loaded: inventory       | Shape: (60247, 17)
Loaded: order_items     | Shape: (714669, 7)
Loaded: web_traffic     | Shape: (3652, 7)
Loaded: shipments       | Shape: (566067, 4)
Loaded: sales           | Shape: (3833, 3)
Loaded: reviews         | Shape: (113551, 7)
Loaded: orders          | Shape: (646945, 8)


In [ ]:
# ----------------------------------------------------------------------------
# 2. DATA TYPE CASTING
# ----------------------------------------------------------------------------
print("\n[INFO] Casting data types...")

date_columns = {
    'orders': ['order_date'],
    'customers': ['signup_date'],
    'reviews': ['review_date'],
    'web_traffic': ['date'],
    'shipments': ['ship_date', 'delivery_date'],
    'sales': ['Date'],
    'promotions': ['start_date', 'end_date'],
    'returns': ['return_date'],
    'inventory': ['snapshot_date']
}

for sheet, cols in date_columns.items():
    if sheet in dfs:
        for col in cols:
            if col in dfs[sheet].columns:
                dfs[sheet][col] = pd.to_datetime(dfs[sheet][col], errors='coerce')

numeric_conversions = {
    'products': ['price', 'cogs'],
    'payments': ['payment_value', 'installments'],
    'order_items': ['quantity', 'unit_price', 'discount_amount'],
    'returns': ['return_quantity', 'refund_amount'],
    'shipments': ['shipping_fee'],
    'sales': ['Revenue', 'COGS'],
    'web_traffic': ['sessions', 'bounce_rate', 'avg_session_duration_sec'],
    'inventory': ['stock_on_hand', 'units_received', 'units_sold', 'stockout_days',
                  'days_of_supply', 'fill_rate', 'sell_through_rate']
}

for sheet, cols in numeric_conversions.items():
    if sheet in dfs:
        for col in cols:
            if col in dfs[sheet].columns:
                dfs[sheet][col] = pd.to_numeric(dfs[sheet][col], errors='coerce')


[INFO] Casting data types...


In [ ]:
# ----------------------------------------------------------------------------
# 3.1. MISSING VALUE INSPECTION
# Scans uncleaned datasets and reports columns with missing values (count and percentage).
# ----------------------------------------------------------------------------

sheet_names = [
    'returns', 'promotions', 'products', 'payments', 'customers',
    'geography', 'inventory', 'order_items', 'web_traffic',
    'shipments', 'sales', 'reviews', 'orders'
]

print("[INFO] GENERATING MISSING VALUES REPORT (RAW DATA)\n" + "="*60)

for sheet in sheet_names:
    file_path = f"{INPUT_DIR}{sheet}.csv"

    if os.path.exists(file_path):
        df = pd.read_csv(file_path, low_memory=False)
        total_rows = len(df)

        # Calculate missing counts and percentages
        missing_count = df.isnull().sum()
        missing_percent = (missing_count / total_rows) * 100

        # Create a summary dataframe and filter only columns with missing data
        summary_df = pd.DataFrame({
            'Missing Count': missing_count,
            'Percentage (%)': missing_percent
        })
        summary_df = summary_df[summary_df['Missing Count'] > 0]

        # Print results
        if not summary_df.empty:
            print(f"\nDataset: {sheet.upper()} | Total Rows: {total_rows:,}")
            print("-" * 40)
            print(summary_df.round(2).to_string())
        else:
            print(f"\nDataset: {sheet.upper()} | Total Rows: {total_rows:,}")
            print("-" * 40)
            print("Status: 0 missing values")

    else:
        print(f"\n[WARNING] File not found: {file_path}")

[INFO] GENERATING MISSING VALUES REPORT (RAW DATA)

Dataset: RETURNS | Total Rows: 39,939
----------------------------------------
Status: 0 missing values

Dataset: PROMOTIONS | Total Rows: 50
----------------------------------------
                     Missing Count  Percentage (%)
applicable_category             40            80.0

Dataset: PRODUCTS | Total Rows: 2,412
----------------------------------------
Status: 0 missing values

Dataset: PAYMENTS | Total Rows: 646,945
----------------------------------------
Status: 0 missing values

Dataset: CUSTOMERS | Total Rows: 121,930
----------------------------------------
Status: 0 missing values

Dataset: GEOGRAPHY | Total Rows: 39,948
----------------------------------------
Status: 0 missing values

Dataset: INVENTORY | Total Rows: 60,247
----------------------------------------
Status: 0 missing values

Dataset: ORDER_ITEMS | Total Rows: 714,669
----------------------------------------
            Missing Count  Percentage (%)
pr

In [ ]:
# ----------------------------------------------------------------------------
# 3.2. MISSING VALUE IMPUTATION
# ----------------------------------------------------------------------------
print("\n[INFO] Handling missing values...")
# 1. For order_items: Enter “none” for products that do not have any promotion applied
if 'order_items' in dfs:
    if 'promo_id' in dfs['order_items'].columns:
        dfs['order_items']['promo_id'] = dfs['order_items']['promo_id'].fillna('none')
    if 'promo_id_2' in dfs['order_items'].columns:
        dfs['order_items']['promo_id_2'] = dfs['order_items']['promo_id_2'].fillna('none')
    print("Imputed 'order_items' (promo_id, promo_id_2) -> filled with 'none'")

# 2. For promotions: Enter “all” because “null” means it applies to all categories
if 'promotions' in dfs:
    if 'applicable_category' in dfs['promotions'].columns:
        dfs['promotions']['applicable_category'] = dfs['promotions']['applicable_category'].fillna('all')
    print("Imputed 'promotions' (applicable_category) -> filled with 'all'")


[INFO] Handling missing values...
Imputed 'order_items' (promo_id, promo_id_2) -> filled with 'none'
Imputed 'promotions' (applicable_category) -> filled with 'all'


In [ ]:
# ----------------------------------------------------------------------------
# 4. CLEANING & VALIDATION
# ----------------------------------------------------------------------------
print("\n[INFO] Handling duplicates and unstandardized data")
for name, df in dfs.items():
    dfs[name] = df.drop_duplicates()
    str_cols = df.select_dtypes(include=['object']).columns
    for col in str_cols:
        dfs[name][col] = dfs[name][col].astype(str).str.lower().str.strip()

print("Duplicates removed and strings standardized")


[INFO] Handling duplicates and unstandardized data
Duplicates removed and strings standardized


In [ ]:
# ----------------------------------------------------------------------------
# 5.1. LOGICAL VALIDATION INSPECTION
# Checks for business logic violations in the raw data (e.g., negative prices, logical sequencing)
# ----------------------------------------------------------------------------

print("[INFO] GENERATING LOGICAL VALIDATION REPORT\n" + "="*80)

# Helper function to safely load data
def load_data(sheet):
    file_path = f"{INPUT_DIR}{sheet}.csv"
    if os.path.exists(file_path):
        return pd.read_csv(file_path, low_memory=False)
    return None

# Load required tables
products = load_data('products')
payments = load_data('payments')
order_items = load_data('order_items')
returns = load_data('returns')
shipments = load_data('shipments')
sales = load_data('sales')
inventory = load_data('inventory')

validation_results = []

def check_constraint(table_name, df, condition, description):
    if df is not None:
        violations = df[condition].shape[0]
        if violations > 0:
            validation_results.append({
                'Table': table_name.upper(),
                'Issue': description,
                'Violations': violations,
                '% of Total': (violations / len(df)) * 100
            })

# A. PRODUCTS LOGIC
if products is not None:
    products['price'] = pd.to_numeric(products['price'], errors='coerce')
    products['cogs'] = pd.to_numeric(products['cogs'], errors='coerce')
    check_constraint('products', products, products['price'] <= 0, 'Price is <= 0')
    check_constraint('products', products, products['cogs'] < 0, 'COGS is negative')
    # Constraint explicitly mentioned in the rule: cogs < price
    check_constraint('products', products, products['cogs'] >= products['price'], 'COGS is >= Price')

# B. PAYMENTS LOGIC
if payments is not None:
    payments['payment_value'] = pd.to_numeric(payments['payment_value'], errors='coerce')
    check_constraint('payments', payments, payments['payment_value'] <= 0, 'Payment value is <= 0')

# C. ORDER ITEMS LOGIC
if order_items is not None:
    order_items['quantity'] = pd.to_numeric(order_items['quantity'], errors='coerce')
    order_items['unit_price'] = pd.to_numeric(order_items['unit_price'], errors='coerce')
    order_items['discount_amount'] = pd.to_numeric(order_items['discount_amount'], errors='coerce')
    check_constraint('order_items', order_items, order_items['quantity'] <= 0, 'Quantity is <= 0')
    check_constraint('order_items', order_items, order_items['unit_price'] < 0, 'Unit price is negative')
    check_constraint('order_items', order_items, order_items['discount_amount'] < 0, 'Discount amount is negative')
    check_constraint('order_items', order_items, order_items['discount_amount'] > (order_items['unit_price'] * order_items['quantity']), 'Discount exceeds total item value')

# D. RETURNS LOGIC
if returns is not None:
    returns['return_quantity'] = pd.to_numeric(returns['return_quantity'], errors='coerce')
    returns['refund_amount'] = pd.to_numeric(returns['refund_amount'], errors='coerce')
    check_constraint('returns', returns, returns['return_quantity'] <= 0, 'Return quantity is <= 0')
    check_constraint('returns', returns, returns['refund_amount'] < 0, 'Refund amount is negative')

# E. SHIPMENTS LOGIC
if shipments is not None:
    shipments['shipping_fee'] = pd.to_numeric(shipments['shipping_fee'], errors='coerce')
    check_constraint('shipments', shipments, shipments['shipping_fee'] < 0, 'Shipping fee is negative')

    # Check logical dates: delivery_date should be >= ship_date
    shipments['ship_date'] = pd.to_datetime(shipments['ship_date'], errors='coerce')
    shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'], errors='coerce')
    check_constraint('shipments', shipments, shipments['delivery_date'] < shipments['ship_date'], 'Delivery date is before Ship date')

# F. SALES & INVENTORY LOGIC
if sales is not None:
    sales['Revenue'] = pd.to_numeric(sales['Revenue'], errors='coerce')
    sales['COGS'] = pd.to_numeric(sales['COGS'], errors='coerce')
    check_constraint('sales', sales, sales['Revenue'] < 0, 'Revenue is negative')
    check_constraint('sales', sales, sales['COGS'] < 0, 'COGS is negative')

if inventory is not None:
    inventory['stock_on_hand'] = pd.to_numeric(inventory['stock_on_hand'], errors='coerce')
    check_constraint('inventory', inventory, inventory['stock_on_hand'] < 0, 'Stock on hand is negative')

# Output Results
if validation_results:
    summary_df = pd.DataFrame(validation_results)
    print(summary_df.round(3).to_string(index=False))
else:
    print("Status: ALL DATA PASSED LOGICAL CONSTRAINTS.")

print("\n" + "="*80)
print("[INFO] LOGICAL INSPECTION COMPLETE.")

[INFO] GENERATING LOGICAL VALIDATION REPORT
Status: ALL DATA PASSED LOGICAL CONSTRAINTS.

[INFO] LOGICAL INSPECTION COMPLETE.


In [ ]:
# ----------------------------------------------------------------------------
# 6. OUTLIER DETECTION SCRIPT (IQR METHOD)
# Scans numerical columns and reports outlier counts and percentages.
# ----------------------------------------------------------------------------

numeric_cols_to_check = {
    'products': ['price', 'cogs'],
    'payments': ['payment_value', 'installments'],
    'order_items': ['quantity', 'unit_price', 'discount_amount'],
    'returns': ['return_quantity', 'refund_amount'],
    'shipments': ['shipping_fee'],
    'sales': ['Revenue', 'COGS'],
    'web_traffic': ['sessions', 'avg_session_duration_sec'],
    'inventory': ['stock_on_hand', 'units_received', 'units_sold']
}

print("[INFO] GENERATING OUTLIER REPORT (IQR METHOD)\n" + "="*80)

for sheet, cols in numeric_cols_to_check.items():
    file_path = f"{INPUT_DIR}{sheet}.csv"

    if os.path.exists(file_path):
        df = pd.read_csv(file_path, low_memory=False)
        total_rows = len(df)
        outlier_data = []

        for col in cols:
            if col in df.columns:
                # Convert to numeric just in case, drop NaNs for calculation
                series = pd.to_numeric(df[col], errors='coerce').dropna()

                if len(series) == 0:
                    continue

                # Calculate Q1, Q3, and IQR
                Q1 = series.quantile(0.25)
                Q3 = series.quantile(0.75)
                IQR = Q3 - Q1

                # Define bounds
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                # Identify outliers
                outliers = series[(series < lower_bound) | (series > upper_bound)]
                outlier_count = len(outliers)
                outlier_percent = (outlier_count / total_rows) * 100

                if outlier_count > 0:
                    outlier_data.append({
                        'Column': col,
                        'Outliers Count': outlier_count,
                        'Percentage (%)': outlier_percent,
                        'Lower Bound': lower_bound,
                        'Upper Bound': upper_bound
                    })

        if outlier_data:
            print(f"\nDataset: {sheet.upper()} | Total Rows: {total_rows:,}")
            print("-" * 80)
            summary_df = pd.DataFrame(outlier_data)
            print(summary_df.round(2).to_string(index=False))
        else:
            print(f"\nDataset: {sheet.upper()} | Total Rows: {total_rows:,}")
            print("-" * 80)
            print("Status: NO OUTLIERS")

    else:
        print(f"\n[WARNING] File not found: {file_path}")

print("\n" + "="*80)
print("[INFO] OUTLIER INSPECTION COMPLETE.")

[INFO] GENERATING OUTLIER REPORT (IQR METHOD)

Dataset: PRODUCTS | Total Rows: 2,412
--------------------------------------------------------------------------------
Column  Outliers Count  Percentage (%)  Lower Bound  Upper Bound
 price              31            1.29    -11432.16     19212.12
  cogs              37            1.53     -8709.71     14609.69

Dataset: PAYMENTS | Total Rows: 646,945
--------------------------------------------------------------------------------
       Column  Outliers Count  Percentage (%)  Lower Bound  Upper Bound
payment_value           30219            4.67    -31356.87     72744.28

Dataset: ORDER_ITEMS | Total Rows: 714,669
--------------------------------------------------------------------------------
         Column  Outliers Count  Percentage (%)  Lower Bound  Upper Bound
     unit_price            8623            1.21     -6143.42     15324.06
discount_amount          105767           14.80     -1451.44      2419.07

Dataset: RETURNS | Total 

In [ ]:
# ----------------------------------------------------------------------------
# 7. EXPORT CLEANED DATA
# ----------------------------------------------------------------------------
print("\n[INFO] Exporting cleaned datasets...")


for sheet, df in dfs.items():
    output_filepath = f"{OUTPUT_DIR}{sheet}.csv"
    df.to_csv(output_filepath, index=False)


[INFO] Exporting cleaned datasets...


In [ ]:
!zip -r datathon_cleaned_data.zip data/cleaned-data/

updating: data/cleaned-data/ (stored 0%)
updating: data/cleaned-data/payments.csv (deflated 71%)
updating: data/cleaned-data/sales.csv (deflated 58%)
updating: data/cleaned-data/promotions.csv (deflated 80%)
updating: data/cleaned-data/products.csv (deflated 70%)
updating: data/cleaned-data/customers.csv (deflated 83%)
updating: data/cleaned-data/inventory.csv (deflated 85%)
updating: data/cleaned-data/shipments.csv (deflated 81%)
updating: data/cleaned-data/orders.csv (deflated 82%)
updating: data/cleaned-data/geography.csv (deflated 89%)
updating: data/cleaned-data/returns.csv (deflated 73%)
updating: data/cleaned-data/order_items.csv (deflated 73%)
updating: data/cleaned-data/web_traffic.csv (deflated 68%)
updating: data/cleaned-data/reviews.csv (deflated 76%)
